## workspace×日次の利用量（＝「workspaceごとの利用料」）

目的：workspaceごとのDBU推移・ランキング

**スキーマ例**

```
usage_date DATE          # 利用基準日（日単位）
workspace_id LONG        # Databricks Workspace識別子
total_usage_quantity DOUBLE  # そのworkspaceの1日合計DBU
active_users LONG        # そのworkspaceを利用したユニークユーザー数
usage_record_count LONG  # usageレコード件数（利用イベント数）
```

In [0]:
%run ../../変数設定

In [0]:
gold_table_name = "gold_usage_workspace_daily"
gold_table_path = f"{catalog_name}.{schema_name}.{gold_table_name}"


In [0]:
spark.sql(f"DROP TABLE IF EXISTS {gold_table_path}")

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {gold_table_path} (
    usage_date DATE,
    workspace_id STRING,
    total_usage_quantity DOUBLE,
    active_users LONG,
    usage_record_count LONG
    )
    PARTITIONED BY (usage_date)
    """
)

In [0]:
# 日別ユーザー集計
df = spark.sql(
    f"""
    SELECT
        DATE(usage_start_time) AS usage_date,
        workspace_id,
        SUM(usage_quantity) AS total_usage_quantity,
        COUNT(DISTINCT email) AS active_users,
        COUNT(*) AS usage_record_count
    FROM my_lab.handson.silver_usage
    GROUP BY DATE(usage_start_time), workspace_id
    """
)

In [0]:
# テーブルとして保存
df.write.mode("overwrite").saveAsTable(gold_table_path)

print("✅ gold_daily_stats テーブルを作成しました")

In [0]:
# 結果を確認
display(spark.table(gold_table_path))
